In [1]:
from sklearn.metrics import make_scorer, f1_score, precision_score


results_all = {}

score = {
    'Accuracy': 'accuracy',
    'F1': make_scorer(f1_score, average='binary'),
    'ROC-AUC': 'roc_auc',
    'Precision': make_scorer(precision_score, average='binary'),
}

In [ ]:
from core import (
    skf,
    X_train,
    Y_train,
    evaluate_model,
    CAT_FEATURES,
    score
)

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV


cat = CatBoostClassifier(
    random_state=42,
    verbose=0,
    thread_count=-1,
    eval_metric='AUC',
    od_type='Iter',
    od_wait=50
)

param_dist_cat = {
    'iterations': [200, 500, 800, 1000, 1500, 2000, 2500, 3000],
    'depth': [3, 4, 5, 6, 7, 8, 9, 10, 12],
    'learning_rate': [0.002, 0.005, 0.008, 0.01, 0.03, 0.05, 0.08, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7, 10, 15, 20, 30, 50],
    'bagging_temperature': [0, 0.5, 1, 1.5, 2],
    'random_strength': [1, 2, 3, 5, 10],
    'border_count': [32, 64, 128, 254],
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'min_data_in_leaf': [1, 3, 5, 10, 20],
}

random_cat = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=200,
    cv=skf,
    scoring='roc_auc',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_cat.fit(X_train, Y_train, cat_features=CAT_FEATURES)

best_cat = random_cat.best_estimator_

print(f'Лучшие параметры CatBoost: {random_cat.best_params_}')
print(f'Лучшая ROC‑AUC: {random_cat.best_score_:.4f}')

results_cat = evaluate_model(
    model=best_cat,
    X=X_train,
    y=Y_train,
    cv=skf,
    scoring_dict=score
)

results_all = {'CatBoost': results_cat}

Fitting 5 folds for each of 200 candidates, totalling 1000 fits


In [ ]:
results_all